In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact

In [ ]:
# --- IPR Data ---
ipr = pd.DataFrame({
    'Pwf': [3000, 2800, 2600, 2130, 1500, 1000, 500, 0],
    'Q':   [0, 100, 200, 435, 709, 867, 973, 1027]
})

# --- TPR Data ---
q = np.array([100, 200, 300, 400, 500, 600, 700, 800, 900, 1000])
p_190 = np.array([1200,1300,1387,1492,1612,1743,1884,2032,2187,2346])
p_2375 = np.array([1168,1220,1251,1290,1335,1387,1445,1509,1577,1649])
p_2875 = np.array([1156,1184,1205,1219,1236,1256,1278,1303,1331,1361])

tpr_base = {
    "1.90\"": p_190,
    "2.375\"": p_2375,
    "2.875\"": p_2875
}


In [ ]:
# Interpolation function
def interpolate_pressure(q_vals, pwf_vals, q_range):
    return np.interp(q_range, q_vals, pwf_vals)

# --- Plot with Intersection ---
def plot_nodal(wc, gor, skin, tubing_id):
    plt.figure(figsize=(9, 6))

    # Plot IPR
    plt.plot(ipr['Q'], ipr['Pwf'], label="IPR", color='black', linewidth=2)

    # Adjust TPR
    base_tpr = tpr_base[tubing_id]
    adj_factor = 1 + 0.002 * wc + 0.001 * gor + 0.05 * skin
    tpr_adj = base_tpr * adj_factor
    tpr_adj = np.clip(tpr_adj, a_min=0, a_max=None)

    # Plot TPR
    plt.plot(q, tpr_adj, label=f"TPR {tubing_id}", color='blue', linewidth=2)

    # Interpolate
    q_interp = np.linspace(100, 1000, 1000)
    ipr_interp = interpolate_pressure(ipr['Q'], ipr['Pwf'], q_interp)
    tpr_interp = interpolate_pressure(q, tpr_adj, q_interp)

    # Difference and threshold
    diff = np.abs(ipr_interp - tpr_interp)
    min_diff = np.min(diff)
    idx = np.argmin(diff)

    # If curves are close enough (say < 25 psi), mark intersection
    if min_diff < 25:
        q_opt = q_interp[idx]
        p_opt = ipr_interp[idx]
        plt.plot(q_opt, p_opt, 'ro', label=f'Optimum Rate ≈ {q_opt:.1f} Mscf/D\n@ {p_opt:.1f} psia')
    else:
        print("⚠️ No valid intersection found between IPR and TPR curves.")

    # Labels and formatting
    plt.xlabel('Flow Rate (Mscf/D)')
    plt.ylabel('Flowing Bottomhole Pressure (psia)')
    plt.title('Nodal Analysis with Sensitivity & Optimum Rate')
    plt.grid(True)
    plt.legend()
    # plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()


# --- Sliders ---
wc_slider = widgets.IntSlider(value=30, min=0, max=90, step=5, description='Water Cut (%)')
gor_slider = widgets.IntSlider(value=800, min=100, max=2000, step=100, description='GOR (scf/bbl)')
skin_slider = widgets.FloatSlider(value=0, min=-2.0, max=10.0, step=0.5, description='Skin')
tubing_slider = widgets.Dropdown(options=["1.90\"", "2.375\"", "2.875\""], value="2.375\"", description="Tubing ID")

interact(plot_nodal, wc=wc_slider, gor=gor_slider, skin=skin_slider, tubing_id=tubing_slider)